# Descriptive Statistics of the Corpus

In [1]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_LOC = '../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

In [3]:
# to rename the corpus correctly . . . 
def rename(x):
    result = x
    if x.startswith('se'):
        result = 'CORAAL'
        
    if x.startswith('call'):
        result = x.split('-')[0]
    
    if x.startswith('MICASE'):
        result = 'MICASE'
    
    if x.startswith('instruction'):
        result = x.split('-')[-1]
    
    if x.startswith('CABNC'):
        result = 'CABNC'
    
    return result

In [4]:
def grab_all_files(PATH, file_type='.csv'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

In [5]:
fs = grab_all_files(DATA_PATH, '.tsv')
fs

['data/lme_ready/cha-ceda-analysis.tsv',
 'data/lme_ready/candor-ceda-analysis.tsv',
 'data/lme_ready/cha-CABNC-ceda-analysis.tsv',
 'data/lme_ready/grace-ceda-analysis.tsv',
 'data/lme_ready/cha-MICASE-ceda-analysis.tsv',
 'data/lme_ready/cha-coraal-ceda-analysis.tsv']

In [6]:
final_tallies = dict()

In [7]:
turn_cut_off = 5

In [8]:
for f in tqdm(fs):
    stats = []
    print(f)
    df = pd.read_table(f,sep='\t')
    
    if ('/cha-' in f) or ('grace-' in f):
        df['conversation_id'] = ['-'.join(sp.split('-')[:-1]) for sp in tqdm(df['x_speaker'])]
   
    if '/candor-' in f:
        df['conversation_id'] = [sp.replace('.csv', '') for sp in tqdm(df['conversation_id'])]
        df['corpus'] = 'CANDOR'
    
    # else:
    #     if 'conversation_id' in df.columns:
    #         0
    #     else:
    #         df['conversation_id'] = f
    
    df['x_turn_id_'] = df['conversation_id'] + '-' + df['x_turn_id'].astype(str)
    df['y_turn_id_'] = df['conversation_id'] + '-' + df['y_turn_id'].astype(str)
    df['corpus'] = [rename(co) for co in tqdm(df['corpus'])]
    
    after_kth_turn = df['x_turn_id'] >= turn_cut_off
    appropriate_n_tokens = (df['nx'] >= 5) & (df['ny'] >= 5)
    
    for corpus in df['corpus'].unique():
        corpus_sel = df['corpus'].isin([corpus])
        
        if corpus in final_tallies.keys():
            ##### all data
            final_tallies[corpus]['all data']['files'] += [f]
            final_tallies[corpus]['all data']['speakers'] = np.concatenate([
                final_tallies[corpus]['all data']['speakers'],
                np.unique(df[['x_speaker', 'y_speaker']].loc[corpus_sel].values)
            ])
            final_tallies[corpus]['all data']['conversations'] = np.concatenate([
                final_tallies[corpus]['all data']['conversations'],
                df['conversation_id'].loc[corpus_sel].unique()
                
            ])
            final_tallies[corpus]['all data']['utterances'] += len(np.unique(df[['x_turn_id_', 'y_turn_id_']].loc[corpus_sel].values))
            final_tallies[corpus]['all data']['comparisons'] += len(df.loc[corpus_sel])
            
            avg_turn_length = df['conversation_id'].loc[corpus_sel].value_counts()
            med_turn_length, avg_turn_length, std_turn_length, turn_n = avg_turn_length.median(), avg_turn_length.mean(), avg_turn_length.std(), len(avg_turn_length)
            
            final_tallies[corpus]['all data']['mean_turn_length'] = (final_tallies[corpus]['all data']['mean_turn_length'] + avg_turn_length)/2
            final_tallies[corpus]['all data']['median_turn_length'] = int((final_tallies[corpus]['all data']['median_turn_length'] + med_turn_length)/2)
            final_tallies[corpus]['all data']['turn_length_std'] = (((final_tallies[corpus]['all data']['turn_length_std']**2) * (final_tallies[corpus]['all data']['conversation_n']-1)) + (std_turn_length * (turn_n-1))) / (final_tallies[corpus]['all data']['conversation_n'] + turn_n - 2)
            final_tallies[corpus]['all data']['turn_length_std'] = np.sqrt(final_tallies[corpus]['all data']['turn_length_std'])
            final_tallies[corpus]['all data']['conversation_n'] += turn_n
            
            ##### after fifth turn
            final_tallies[corpus]['after fifth turn']['files'] += [f]
            final_tallies[corpus]['after fifth turn']['speakers'] = np.concatenate([
                final_tallies[corpus]['after fifth turn']['speakers'],
                np.unique(df[['x_speaker', 'y_speaker']].loc[corpus_sel & after_kth_turn].values)
            ])
            final_tallies[corpus]['after fifth turn']['conversations'] = np.concatenate([
                final_tallies[corpus]['after fifth turn']['conversations'],
                df['conversation_id'].loc[corpus_sel & after_kth_turn].unique()
                
            ])
            final_tallies[corpus]['after fifth turn']['utterances'] += len(np.unique(df[['x_turn_id_', 'y_turn_id_']].loc[corpus_sel & after_kth_turn].values))
            final_tallies[corpus]['after fifth turn']['comparisons'] += len(df.loc[corpus_sel & after_kth_turn])
            
            
            
            avg_turn_length = df['conversation_id'].loc[corpus_sel & after_kth_turn].value_counts()
            med_turn_length, avg_turn_length, std_turn_length, turn_n = avg_turn_length.median(), avg_turn_length.mean(), avg_turn_length.std(), len(avg_turn_length)
            
            final_tallies[corpus]['after fifth turn']['mean_turn_length'] = (final_tallies[corpus]['after fifth turn']['mean_turn_length'] + avg_turn_length)/2
            final_tallies[corpus]['after fifth turn']['median_turn_length'] = int((final_tallies[corpus]['after_fift_turn']['median_turn_length'] + med_turn_length)/2)
            final_tallies[corpus]['after fifth turn']['turn_length_std'] = (((final_tallies[corpus]['after fifth turn']['turn_length_std']**2) * (final_tallies[corpus]['after fifth turn']['conversation_n']-1)) + (std_turn_length * (turn_n-1))) / (final_tallies[corpus]['after fifth turn']['conversation_n'] + turn_n - 2)
            final_tallies[corpus]['after fifth turn']['turn_length_std'] = np.sqrt(final_tallies[corpus]['after fifth turn']['turn_length_std'])
            final_tallies[corpus]['after fifth turn']['conversation_n'] += turn_n
            
            
            ##### fully parsed
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['files'] += [f]
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['speakers'] = np.concatenate([
                final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['speakers'],
                np.unique(df[['x_speaker', 'y_speaker']].loc[corpus_sel & after_kth_turn & appropriate_n_tokens].values)
            ])
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['conversations'] = np.concatenate([
                final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['conversations'],
                df['conversation_id'].loc[corpus_sel & after_kth_turn & appropriate_n_tokens].unique()
                
            ])
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['utterances'] += len(np.unique(df[['x_turn_id_', 'y_turn_id_']].loc[corpus_sel & after_kth_turn & appropriate_n_tokens].values))
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['comparisons'] += len(df.loc[corpus_sel & after_kth_turn & appropriate_n_tokens])
            
            
            
            avg_turn_length = df['conversation_id'].loc[corpus_sel & after_kth_turn & appropriate_n_tokens].value_counts()
            avg_turn_length, std_turn_length, turn_n = avg_turn_length.mean(), avg_turn_length.std(), len(avg_turn_length)
            
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['mean_turn_length'] = (final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['mean_turn_length'] + avg_turn_length)/2
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['median_turn_length'] = int((final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['median_turn_length'] + med_turn_length)/2)
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['turn_length_std'] = (((final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['turn_length_std']**2) * (final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['conversation_n']-1)) + (std_turn_length * (turn_n-1))) / (final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['conversation_n'] + turn_n - 2)
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['turn_length_std'] = np.sqrt(final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['turn_length_std'])
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['conversation_n'] += turn_n
            
                
        else:
            final_tallies[corpus] = {
                # all data
                'all data': {
                    'files': [f],
                    'split': 'all data',
                    'speakers': np.unique(df[['x_speaker', 'y_speaker']].loc[corpus_sel].values),
                    'conversations': df['conversation_id'].loc[corpus_sel].unique(),
                    'utterances': len(np.unique(df[['x_turn_id_', 'y_turn_id_']].loc[corpus_sel].values)),
                    'comparisons': len(df.loc[corpus_sel])
                },
                
                # after fifth turn
                'after fifth turn': {
                    'files': [f],
                    'split': 'after fifth turn',
                    'speakers': np.unique(df[['x_speaker', 'y_speaker']].loc[corpus_sel & after_kth_turn].values),
                    'conversations': df['conversation_id'].loc[corpus_sel & after_kth_turn].unique(),
                    'utterances': len(np.unique(df[['x_turn_id_', 'y_turn_id_']].loc[corpus_sel & after_kth_turn].values)),
                    'comparisons': len(df.loc[corpus_sel & after_kth_turn])
                },
                
                # after fifth turn & n >= 5
                'after fifth turn & $n_{tokens} \geq 5$': {
                    'files': [f],
                    'split': 'after fifth turn & $n_{tokens} \geq 5$',
                    'speakers': np.unique(df[['x_speaker', 'y_speaker']].loc[corpus_sel & after_kth_turn & appropriate_n_tokens].values),
                    'conversations': df['conversation_id'].loc[corpus_sel & after_kth_turn & appropriate_n_tokens].unique(),
                    'utterances': len(np.unique(df[['x_turn_id_', 'y_turn_id_']].loc[corpus_sel & after_kth_turn & appropriate_n_tokens].values)),
                    'comparisons': len(df.loc[corpus_sel & after_kth_turn & appropriate_n_tokens])
                }
            }
            
            avg_turn_length = df['conversation_id'].loc[corpus_sel].value_counts()
            med_turn_length, avg_turn_length, std_turn_length, turn_n = avg_turn_length.median(), avg_turn_length.mean(), avg_turn_length.std(), len(avg_turn_length)
            
            final_tallies[corpus]['all data']['mean_turn_length'] = avg_turn_length
            final_tallies[corpus]['all data']['median_turn_length'] = med_turn_length
            final_tallies[corpus]['all data']['turn_length_std'] = std_turn_length
            final_tallies[corpus]['all data']['conversation_n'] = turn_n
            
            avg_turn_length = df['conversation_id'].loc[corpus_sel & after_kth_turn & appropriate_n_tokens].value_counts()
            med_turn_length, avg_turn_length, std_turn_length, turn_n = avg_turn_length.median(), avg_turn_length.mean(), avg_turn_length.std(), len(avg_turn_length)
            
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['mean_turn_length'] = avg_turn_length
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['median_turn_length'] = med_turn_length
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['turn_length_std'] = std_turn_length
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['conversation_n'] = turn_n
            
            avg_turn_length = df['conversation_id'].loc[corpus_sel & after_kth_turn].value_counts()
            med_turn_length, avg_turn_length, std_turn_length, turn_n = avg_turn_length.median(), avg_turn_length.mean(), avg_turn_length.std(), len(avg_turn_length)
            
            final_tallies[corpus]['after fifth turn']['mean_turn_length'] = avg_turn_length
            final_tallies[corpus]['after fifth turn']['median_turn_length'] = med_turn_length
            final_tallies[corpus]['after fifth turn']['turn_length_std'] = std_turn_length
            final_tallies[corpus]['after fifth turn']['conversation_n'] = turn_n
    
    del df

  0%|          | 0/6 [00:00<?, ?it/s]

data/lme_ready/cha-ceda-analysis.tsv


  0%|          | 0/8028150 [00:00<?, ?it/s]

  0%|          | 0/8028150 [00:00<?, ?it/s]

data/lme_ready/candor-ceda-analysis.tsv


...

  0%|          | 0/8262622 [00:00<?, ?it/s]

  0%|          | 0/8262622 [00:00<?, ?it/s]

data/lme_ready/cha-coraal-ceda-analysis.tsv


  0%|          | 0/11936993 [00:00<?, ?it/s]

  0%|          | 0/11936993 [00:00<?, ?it/s]

In [9]:
stats = []

In [10]:
for corpus, splits in final_tallies.items():
    for k,v in splits.items():
        stats += [v]
        stats[-1]['corpus'] = corpus

In [11]:
stats = pd.DataFrame(stats)

In [12]:
stats['speakers'] = [len(np.unique(sp)) for sp in stats['speakers']]
stats['conversations'] = [len(np.unique(sp)) for sp in stats['conversations']]

In [13]:
stats.head(n=100)

,files,split,speakers,conversations,utterances,comparisons,mean_turn_length,median_turn_length,turn_length_std,conversation_n,corpus
0,[data/lme_ready/cha-ceda-analysis.tsv],all data,106,40,4881,217359,5433.975000,4823.5,2255.438152,40,GCSAusE
1,[data/lme_ready/cha-ceda-analysis.tsv],after fifth turn,106,40,4722,207644,5191.100000,4575.5,2254.010088,40,GCSAusE
2,[data/lme_ready/cha-ceda-analysis.tsv],after fifth turn & $n_{tokens} \geq 5$,106,40,3693,127830,3195.750000,3098.0,1379.617687,40,GCSAusE
3,[data/lme_ready/cha-ceda-analysis.tsv],all data,341,60,45214,2599832,43330.533333,40074.0,21134.801397,60,SBCSAE
4,[data/lme_ready/cha-ceda-analysis.tsv],after fifth turn,337,60,44974,2585651,43094.183333,39866.5,21128.707975,60,SBCSAE
5,[data/lme_ready/cha-ceda-analysis.tsv],after fifth turn & $n_{tokens} \geq 5$,320,60,33140,1385635,23093.916667,20049.5,10715.904480,60,SBCSAE
6,[data/lme_ready/cha-ceda-analysis.tsv],all data,87,32,10585,560676,17521.125000,1525.5,40882.043337,32,SCoSE
7,[data/lme_ready/cha-ceda-analysis.tsv],after fifth turn,85,31,10458,555321,17913.580645,1661.0,41401.611290,31,SCoSE
8,[data/lme_ready/cha-ceda-analysis.tsv],after fifth turn & $n_{tokens} \geq 5$,79,31,8002,316582,10212.322581,1377.0,20425.204514,31,SCoSE
9,[data/lme_ready/cha-ceda-analysis.tsv],all data,93,40,23444,1367343,34183.575000,21468.0,31432.080574,40,callfriend


In [15]:
stats.to_csv(os.path.join(DATA_LOC,'corpus-descriptives.csv'), index=False, encoding='utf-8')

## Other visualizations

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.figure_factory as ff

In [ ]:
# df = 